# MEEP Mode Solver - Rib Waveguide

Compute the fundamental TE mode of a **silicon rib waveguide** at a
single wavelength, returning `n_eff` and the full 2D (X, Z) mode
profile.

**Rib cross-section:**
 - SiO2 box (2 um)
 - Si slab (70 nm, partially etched)
 - Si rib (150 nm on top of slab -> total 220 nm)

The rib provides lateral confinement via effective-index contrast
between the ridge and the thinner side slabs.

### Imports & MEEP check

In [ ]:
import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np

from gsim.common.stack.extractor import Layer, LayerStack
from gsim.meep import (
    mode_x_grid,
    mode_z_grid,
    refractive_index_profile,
    solve_cross_section_mode,
)

try:
    import meep as mp

    print(f"MEEP {mp.__version__} ready")
except ImportError as err:
    raise SystemExit(
        "MEEP not found. Install it via conda-forge:\n"
        "    conda install -c conda-forge pymeep"
    ) from err

### Build the GDS component

We manually create a component with **two Si layers** - a wide slab and
a narrow rib on top.  The mode solver later slices through both layers
at the port Y-position to reconstruct the 2D (X,Z) geometry.

In [ ]:
SLAB_WIDTH = 2.0  # um
RIB_WIDTH = 0.5  # um
LENGTH = 10.0  # um

c = gf.Component(name="rib_waveguide")

# Si slab layer (layer 1) - wide, partially etched
c.add_polygon(
    [
        (-LENGTH / 2, -SLAB_WIDTH / 2),
        (LENGTH / 2, -SLAB_WIDTH / 2),
        (LENGTH / 2, SLAB_WIDTH / 2),
        (-LENGTH / 2, SLAB_WIDTH / 2),
    ],
    layer=(1, 0),
)

# Si rib layer (layer 2) - narrow ridge on top
c.add_polygon(
    [
        (-LENGTH / 2, -RIB_WIDTH / 2),
        (LENGTH / 2, -RIB_WIDTH / 2),
        (LENGTH / 2, RIB_WIDTH / 2),
        (-LENGTH / 2, RIB_WIDTH / 2),
    ],
    layer=(2, 0),
)

# Ports at both ends
c.add_port(
    name="o1", center=(-LENGTH / 2, 0), width=SLAB_WIDTH, orientation=180, layer=(1, 0)
)
c.add_port(
    name="o2", center=(LENGTH / 2, 0), width=SLAB_WIDTH, orientation=0, layer=(1, 0)
)

print(f"Component: {c.name}")
print(f"  Ports:  {[p.name for p in c.ports]}")
print(f"  Layers: {list(c.layers)}")

### Layer stack

Define the vertical material profile.  Each non-air layer maps a GDS
layer to a Z-range.

In [ ]:
layers = {
    "ox": Layer(
        name="box",
        gds_layer=(0, 0),
        zmin=-2.0,
        zmax=0.0,
        thickness=2.0,
        material="sio2",
        layer_type="dielectric",
    ),
    "slab": Layer(
        name="slab",
        gds_layer=(1, 0),
        zmin=0.0,
        zmax=0.07,
        thickness=0.07,
        material="si",
        layer_type="dielectric",
    ),
    "rib": Layer(
        name="rib",
        gds_layer=(2, 0),
        zmin=0.07,
        zmax=0.22,
        thickness=0.15,
        material="si",
        layer_type="dielectric",
    ),
}
stack = LayerStack(layers=layers)

print("Layer stack:")
for name, l in stack.layers.items():
    print(
        f"  {name:6s}  z=[{l.zmin:+.3f}, {l.zmax:+.3f}]  t={l.thickness:.3f}  material={l.material}"
    )

### Solve the fundamental mode

In [ ]:
WAVELENGTH = 1.55  # um
RESOLUTION = 32

result = solve_cross_section_mode(
    component=c,
    stack=stack,
    port="o1",
    wavelength=WAVELENGTH,
    band_num=1,
    parity="NO_PARITY",
    resolution=RESOLUTION,
)

print(f"n_eff    = {result.n_eff:.6f}")
print(f"n_group  = {result.n_group}")
print(f"kdom     = {[f'{k:.6f}' for k in result.kdom]}")
print(f"band     = {result.band_num}, parity = {result.parity}")
print(f"fields   = {list(result.fields.keys())}")
for comp, arr in result.fields.items():
    print(f"  {comp}: shape={arr.shape}  |max|={np.abs(arr).max():.6f}")

### 2D mode profile - dominant component

The fundamental TE-like rib mode has its primary electric field in Ey.

In [ ]:
dom_comp = max(result.fields, key=lambda k: np.abs(result.fields[k]).max())
field_2d = np.abs(result.fields[dom_comp])

nz, nx = field_2d.shape
x_span = SLAB_WIDTH + 2.0  # port width + margin
z_span = 2.22  # total stack height

x_um = mode_x_grid(n_points=nx, x_span=x_span)
z_um = mode_z_grid(stack, n_points=nz)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: field amplitude
im = ax1.pcolormesh(x_um, z_um, field_2d, shading="auto", cmap="inferno")
plt.colorbar(im, ax=ax1, label=f"|{dom_comp}| (arb. units)")
ax1.set_xlabel("x (um)")
ax1.set_ylabel("z (um)")
ax1.set_title(f"|{dom_comp}|  n_eff={result.n_eff:.4f}")
ax1.set_aspect("equal")

# Right: field with index contours
n_prof = refractive_index_profile(stack, z_um, wavelength=WAVELENGTH)
n_xz = np.tile(n_prof[:, np.newaxis], (1, nx))

im2 = ax2.pcolormesh(x_um, z_um, field_2d, shading="auto", cmap="inferno", alpha=0.85)
plt.colorbar(im2, ax=ax2, label=f"|{dom_comp}| (arb. units)")
ct = ax2.contour(
    x_um, z_um, n_xz, levels=[1.4, 2.0, 3.0], colors="cyan", linewidths=0.8
)
ax2.clabel(ct, fmt="n=%.1f", fontsize=7)
ax2.set_xlabel("x (um)")
ax2.set_ylabel("z (um)")
ax2.set_title("With refractive-index contours")
ax2.set_aspect("equal")

fig.suptitle(
    f"Rib waveguide fundamental TE mode  "
    f"(lambda={WAVELENGTH:.2f} um, slab={SLAB_WIDTH:.1f}um, rib={RIB_WIDTH:.1f}um)",
    fontweight="bold",
)
fig.tight_layout()